# Week 1 Lab — Images as Data (MNIST)

**Goals**
- Load image data and perform basic EDA
- Use PCA as a statistical summary of variation
- Compare a transparent baseline with a flexible predictive baseline
- Start critiquing benchmarks and dataset bias

Breiman's "Two Cultures" is useful background for this week, but we will treat it as a provocation rather than a rigid map. The lab question is: what evidence do different modeling choices give us?

In [ ]:
# --------- INSTALLS (Colab) ----------
# Uncomment if needed (or pin versions later).
# !pip -q install numpy scikit-learn torch torchvision matplotlib plotly seaborn

# --------- IMPORTS ----------
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

import torch
import torchvision
import torchvision.transforms as T

# --------- REPRODUCIBILITY ----------
np.random.seed(0)
torch.manual_seed(0)

# --------- DEVICE ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --------- HELPERS ----------
def show_confusion(y_true, y_pred, labels=None, title="Confusion matrix"):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig = px.imshow(cm, text_auto=True, aspect="auto",
                    labels=dict(x="Predicted", y="True", color="Count"),
                    title=title)
    fig.show()
    return cm

def show_validation_examples(indices, predictions, title, max_images=12):
    """Show selected validation images with true and predicted labels."""
    if len(indices) == 0:
        print("No examples to show.")
        return
    indices = indices[:max_images]
    cols = min(6, len(indices))
    rows = int(np.ceil(len(indices) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, idx in zip(axes, indices):
        ax.imshow(X_val[idx].reshape(28, 28), cmap="gray")
        ax.set_title(f"true={y_val[idx]}, pred={predictions[idx]}")
        ax.axis("off")
    for ax in axes[len(indices):]:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## Week 1 Lab
Images as data + EDA + baseline classifiers (MNIST).

We will compare two baselines:

- **multinomial logistic regression**, a transparent probability model with linear decision boundaries in pixel space
- **random forest**, a more flexible predictive model that can capture nonlinear interactions

The goal is not to declare one model "statistical" and the other "not statistical." The goal is to ask what each model helps us see.

### Part 1 — Load MNIST

**Before you run the next cell:** write down what you expect a single MNIST image to look like as data.

- What are the dimensions of one image?
- What range of values do you expect the pixels to take?
- What information is already missing if all we have is a 28 by 28 grayscale image and a digit label?

In [ ]:
transform = T.Compose([T.ToTensor()])
train_ds = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds  = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

x0, y0 = train_ds[0]
plt.imshow(x0.squeeze(0), cmap="gray")
plt.title(f"MNIST example (label={y0})")
plt.axis("off")
plt.show()


**Pause and interpret.** In 2-3 sentences, describe this object as a statistical observation, not just as a picture.

- What is the feature vector?
- What is the response variable?
- What might the label fail to capture about the handwriting?

### Part 2 — EDA: class balance

**Before you run the plot:** predict whether the class counts will be exactly balanced, approximately balanced, or noticeably imbalanced.

Then answer: why would class balance matter for interpreting accuracy?

In [ ]:
n = 12000
y_train = np.array([train_ds[i][1] for i in range(n)])
counts = np.bincount(y_train, minlength=10)

fig = px.bar(x=list(range(10)), y=counts, labels={"x":"Digit", "y":"Count"}, title="MNIST class balance (subset)")
fig.show()


**Question.** After seeing the plot, would a classifier that always predicts the most common digit be a useful baseline here? Why or why not?

**Claim check.** Complete this sentence: "Because the class distribution is ________, validation accuracy will/will not be strongly distorted by ________."

### Part 3 — Mean/variance images (subset)

**Before you run the next cell:** predict what the mean image and variance image will show.

- Where do you expect the mean image to be brightest?
- Where do you expect pixel variance to be largest?
- What would it mean if the corners had high variance?

In [ ]:
xs = torch.stack([train_ds[i][0] for i in range(n)])  # [n,1,28,28]
mean_img = xs.mean(dim=0).squeeze(0).numpy()
var_img  = xs.var(dim=0).squeeze(0).numpy()

fig, ax = plt.subplots(1,2, figsize=(8,3))
ax[0].imshow(mean_img, cmap="gray"); ax[0].set_title("Mean image"); ax[0].axis("off")
ax[1].imshow(var_img, cmap="magma"); ax[1].set_title("Pixel variance"); ax[1].axis("off")
plt.tight_layout()
plt.show()


**Interpretation.** Write 3-4 sentences connecting these images to the data-generating process.

- What do these plots suggest about centering, scaling, or preprocessing?
- Which pixels seem informative for distinguishing digits?
- Which pixels seem almost irrelevant?
- What would change if the images came from phone photos of handwritten forms instead of curated MNIST scans?

### Part 4 — PCA

**Before you run PCA:** make a prediction.

- Which digit pairs do you expect to overlap most in the first two principal components?
- Which digit pairs do you expect to separate most clearly?
- Why might PCA separate some digits even though it does not use the labels?

In [ ]:
X = xs.view(n, -1).numpy()
pca = PCA(n_components=20, random_state=0)
Z = pca.fit_transform(X)

fig = px.scatter(x=Z[:,0], y=Z[:,1], color=y_train.astype(str),
                 labels={"x":"PC1", "y":"PC2", "color":"Digit"},
                 title="MNIST PCA (PC1 vs PC2)")
fig.show()


**Question.** Pick two digits that overlap in the PCA plot and two digits that separate reasonably well.

For each pair, give a visual or statistical reason. Avoid writing only "the colors overlap"; connect the pattern to handwriting shape, pixel variation, or the limits of projecting 784 dimensions into two.

### Part 5 — Baseline classifiers

Fit two baselines on the same training/validation split. Keep the split fixed so the comparison is paired: both models are tested on the same validation images.

**Before fitting:** which model do you expect to have higher validation accuracy, and why? Your reason should mention the structure of the model, not just "more flexible is better."

In [ ]:
X_train, X_val, y_tr, y_val = train_test_split(X, y_train, test_size=0.2, random_state=0, stratify=y_train)

logit = LogisticRegression(max_iter=200, multi_class="multinomial", solver="lbfgs")
logit.fit(X_train, y_tr)

rf = RandomForestClassifier(n_estimators=100, max_depth=16, random_state=0, n_jobs=-1)
rf.fit(X_train, y_tr)

pred_logit = logit.predict(X_val)
pred_rf = rf.predict(X_val)

acc_logit = accuracy_score(y_val, pred_logit)
acc_rf = accuracy_score(y_val, pred_rf)

print(f"Logistic regression validation accuracy: {acc_logit:.3f}")
print(f"Random forest validation accuracy:       {acc_rf:.3f}")
print(f"Difference (RF - logit):                 {acc_rf - acc_logit:.3f}")


**Interpretation.** Answer in 3-5 sentences.

- Was your prediction about which model would perform better correct?
- Is the difference large enough to matter for the claims you would want to make this week?
- What does logistic regression make easier to explain?
- What does the random forest make easier to capture?
- What does neither accuracy number tell you?

### Part 6 — Error patterns

Accuracy gives one number. Confusion matrices show where that number came from.

**Before you run the confusion matrices:** choose two digit pairs you expect to be confused. Give a handwriting-based reason for each prediction.

In [ ]:
show_confusion(y_val, pred_logit, labels=list(range(10)), title="Logistic regression confusion matrix")
show_confusion(y_val, pred_rf, labels=list(range(10)), title="Random forest confusion matrix")

both_wrong = np.where((pred_logit != y_val) & (pred_rf != y_val))[0]
logit_wrong_rf_right = np.where((pred_logit != y_val) & (pred_rf == y_val))[0]
rf_wrong_logit_right = np.where((pred_rf != y_val) & (pred_logit == y_val))[0]

print("Both wrong:", len(both_wrong))
print("Logit wrong, RF right:", len(logit_wrong_rf_right))
print("RF wrong, logit right:", len(rf_wrong_logit_right))


**Question.** Compare the two confusion matrices.

- Which confusion appears in both models?
- Which confusion seems model-specific?
- If a stakeholder only saw the two accuracy numbers, what important error pattern might they miss?

### Part 7 — Look at individual failures

Now inspect actual validation images. This is a small but important statistical habit: before explaining a model failure, look at the observations.

In [ ]:
show_validation_examples(both_wrong, pred_rf, "Examples both models missed; title uses RF prediction")
show_validation_examples(logit_wrong_rf_right, pred_logit, "Logistic regression wrong, RF right")
show_validation_examples(rf_wrong_logit_right, pred_rf, "RF wrong, logistic regression right")


**Error analysis.** Pick three images from the displays above.

For each one, answer:

- Does the image look ambiguous to you as a human?
- Is the model's mistake understandable from the pixels?
- Does this example suggest a data issue, a model issue, or an inherently ambiguous label?

**Bigger question.** If the true label itself can be ambiguous, what does that imply about treating accuracy as the final measure of performance?

### Final synthesis

Write a short paragraph answering each question.

1. What did EDA teach you that model accuracy alone would not?
2. What did the baseline comparison teach you that a single model would not?
3. Where did looking at individual errors change or complicate your interpretation?
4. What are plausible sources of bias in MNIST as a benchmark?
5. What would you want to know before using MNIST performance as evidence about a deployed digit-recognition system?

## Submission instructions

1. **Run all** cells (Runtime → Run all).
2. Export to PDF:
   - Colab: **File → Print → Save as PDF**.
3. Submit the PDF to Gradescope (**HW## (PDF)**).
4. Optional: download the notebook (`.ipynb`) and submit to **HW## (Notebook – optional)**.
